## YOLOv8n 하이퍼파라미터 튜닝

 셀 1: 라이브러리 import 및 GPU 확인

In [1]:
# ============================================
# YOLOv8n 하이퍼파라미터 튜닝
# 목적: Augmentation 강화로 실제 성능 개선
# ============================================

# YOLO 모델을 사용하기 위한 Ultralytics 라이브러리
from ultralytics import YOLO

# 파일 경로를 다루기 위한 라이브러리
from pathlib import Path

# 시간 측정을 위한 라이브러리
import time

# 현재 날짜/시각을 가져오기 위한 라이브러리
from datetime import datetime

# PyTorch 라이브러리 (GPU 확인용)
import torch

# GPU 확인
print("=" * 60)
print("🔥 YOLOv8n 하이퍼파라미터 튜닝")
print("=" * 60)
print(f"\nCUDA 사용 가능: {torch.cuda.is_available()}")

# GPU가 있으면 정보 출력
if torch.cuda.is_available():
    # GPU 이름 출력
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # GPU 메모리 크기 출력 (GB 단위)
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    # GPU 없으면 CPU 모드 경고
    print("⚠️ CPU 모드로 실행됩니다 (느림)")

print("=" * 60)

🔥 YOLOv8n 하이퍼파라미터 튜닝

CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 4060 Ti
메모리: 8.0 GB


셀 2: 경로 설정

In [2]:
# ============================================
# 경로 설정
# ============================================

# 프로젝트 루트 경로
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 학습 데이터 경로 (70:15 비율로 분할된 데이터)
DATA_DIR = PROJECT_ROOT / 'DATASET' / '5000_split_70_15'

# YOLO 학습용 설정 파일 경로 (train/val 경로, 클래스 정보 포함)
DATA_YAML = DATA_DIR / 'data_70_15.yaml'

# 학습 결과 저장 경로 (하이퍼파라미터 튜닝 버전)
RESULTS_DIR = PROJECT_ROOT / 'results' / 'yolov8n_tuned'

# 경로 정보 출력
print("\n[경로 확인]")
print(f"데이터: {DATA_DIR}")
print(f"YAML:  {DATA_YAML}")
print(f"결과:  {RESULTS_DIR}")

# YAML 파일 존재 여부 확인
if DATA_YAML.exists():
    # 파일이 있으면 OK
    print(f"\n✅ data.yaml 존재")
else:
    # 파일이 없으면 에러 (학습 불가)
    print(f"\n❌ data.yaml 없음!")
    print(f"   경로 확인: {DATA_YAML}")

# Train 폴더 존재 및 이미지 개수 확인
train_dir = DATA_DIR / 'train' / 'images'
if train_dir.exists():
    # Train 이미지 개수 세기
    train_count = len(list(train_dir.glob('*.jpg')))
    print(f"✅ Train: {train_count:,}장")
else:
    # Train 폴더 없으면 에러
    print(f"❌ Train 폴더 없음")

# Validation 폴더 존재 및 이미지 개수 확인
val_dir = DATA_DIR / 'val' / 'images'
if val_dir.exists():
    # Val 이미지 개수 세기
    val_count = len(list(val_dir.glob('*.jpg')))
    print(f"✅ Val:   {val_count:,}장")
else:
    # Val 폴더 없으면 에러
    print(f"❌ Val 폴더 없음")

print("=" * 60)


[경로 확인]
데이터: N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15
YAML:  N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15\data_70_15.yaml
결과:  N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned

✅ data.yaml 존재
✅ Train: 4,172장
✅ Val:   894장


셀 3: 모델 로드

In [3]:
# ============================================
# YOLOv8n 모델 로드
# ============================================

print("\n[모델 로드]")

# YOLOv8n 사전 학습된 가중치 파일 로드
# 'yolov8n.pt'는 Ultralytics에서 제공하는 사전학습 모델
# 처음 실행 시 자동으로 다운로드됨
model = YOLO('yolov8n.pt')

print("✅ YOLOv8n 사전학습 모델 로드 완료")
print("=" * 60)


[모델 로드]
✅ YOLOv8n 사전학습 모델 로드 완료


셀 4: 하이퍼파라미터 튜닝 학습 시작

In [4]:
# ============================================
# 하이퍼파라미터 튜닝 학습 시작
# ============================================

print("\n[하이퍼파라미터 튜닝 학습 시작]")
# 현재 시각 출력 (시작 시간 기록)
print(f"시작 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n【튜닝 내용】")
print("  1. Augmentation 강화 (scale, mosaic, copy_paste, mixup)")
print("  2. 색상 증강 강화 (hsv_h, hsv_s, hsv_v)")
print("  3. 기하학적 증강 추가 (degrees, translate, flipud)")
print("  4. Patience 조정 (50 → 15)")
print("=" * 60)

# 시작 시간 기록 (소요 시간 계산용)
start_time = time.time()

# ============================================
# YOLO 모델 학습 실행 (하이퍼파라미터 튜닝)
# ============================================

results = model.train(

    # ── 기본 설정 ──────────────────────────────────

    # 데이터 설정 파일 경로 (train/val 경로, 클래스 정보)
    data=str(DATA_YAML),

    # 학습 반복 횟수 (데이터 전체를 100번 반복 학습)
    epochs=100,

    # 입력 이미지 크기 (640x640으로 리사이즈)
    imgsz=640,

    # 배치 크기 (한 번에 16장씩 처리) - 메모리 부족 시 8로 변경
    batch=16,

    # 사용할 디바이스 (0=첫번째 GPU, 'cpu'=CPU)
    device=0,

    # 결과 저장 상위 폴더
    project=str(RESULTS_DIR.parent),

    # 결과 저장 폴더명
    name=RESULTS_DIR.name,

    # ── Early Stopping (수정!) ────────────────────

    # 15 에폭 동안 개선 없으면 중단 (50 → 15로 수정)
    # 이유: 50은 너무 과함 (전체의 50%)
    #       15가 적절 (전체의 15%)
    patience=15,

    # ── 저장 및 출력 설정 ──────────────────────────

    # 체크포인트 저장 여부 (best.pt, last.pt 저장)
    save=True,

    # 학습 그래프 생성 여부 (confusion matrix, results.png 등)
    plots=True,

    # Validation 실행 여부 (에폭마다 검증)
    val=True,

    # 상세 출력 여부 (진행 상황 자세히 표시)
    verbose=True,

    # ── 스케일 다양성 증강 (핵심!) ──────────────────

    # [scale] 이미지 내 객체 크기를 랜덤하게 변환하는 비율
    # 기본값: 0.5 → 0.9로 증가
    # 효과: 작은 화재 ~ 큰 화재까지 다양한 크기로 학습
    # 왜: 작은 화재와 큰 화재를 동시에 잘 탐지하려면
    #     다양한 크기의 객체를 많이 봐야 함
    scale=0.9,

    # [mosaic] 4장의 이미지를 1장으로 합쳐서 학습
    # 값: 0.0(끔) ~ 1.0(항상)
    # 기본값: 1.0 유지
    # 효과: 한 이미지에 크고 작은 객체가 동시 등장
    #       스케일 다양성 증가
    mosaic=1.0,

    # [copy_paste] 객체를 잘라 다른 이미지에 붙이기
    # 기본값: 0.0 → 0.3으로 활성화
    # 효과: 화재 객체를 다양한 배경에 배치
    #       데이터 부족 시 매우 효과적
    # 주의: 너무 높으면 (0.5+) 부자연스러움
    copy_paste=0.3,

    # [mixup] 두 이미지를 반투명하게 겹치기
    # 기본값: 0.0 → 0.15로 활성화
    # 효과: 모델이 화재의 본질적 특징에 집중
    #       배경 의존도 감소
    mixup=0.15,

    # ── 기하학적 증강 ─────────────────────────────

    # [degrees] 이미지를 랜덤 회전하는 최대 각도
    # 기본값: 0.0 → 15.0으로 설정
    # 효과: 기울어진 카메라 각도 대응
    #       15도면 충분, 너무 크면 학습 불안정
    degrees=15.0,

    # [translate] 이미지를 상하좌우로 이동하는 비율
    # 기본값: 0.1 → 0.2로 증가
    # 효과: 화재가 이미지 구석에 있는 경우 학습
    #       실제 CCTV는 중앙에만 있지 않음
    translate=0.2,

    # [fliplr] 이미지를 좌우 반전할 확률
    # 기본값: 0.5 유지 (50% 확률)
    # 이유: 화재는 좌우 대칭
    fliplr=0.5,

    # [flipud] 이미지를 상하 반전할 확률
    # 기본값: 0.0 → 0.1로 약하게 활성화
    # 효과: 천장 카메라 등 다양한 시점 대응
    # 주의: 너무 높으면 비현실적 (연기가 아래로)
    flipud=0.1,

    # ── 색상 증강 ─────────────────────────────────

    # [hsv_h] 색조(Hue) 변환 범위
    # 기본값: 0.015 → 0.02로 증가
    # 효과: 화재 색상이 주황~빨강~노랑으로 다양
    #       색조 변화로 다양한 색상 불꽃 대응
    hsv_h=0.02,

    # [hsv_s] 채도(Saturation) 변환 범위
    # 기본값: 0.7 → 0.8로 증가
    # 효과: 연기 많은 화재(채도 낮음) ~
    #       선명한 불꽃(채도 높음) 대응
    hsv_s=0.8,

    # [hsv_v] 명도(Value/Brightness) 변환 범위
    # 기본값: 0.4 → 0.5로 증가
    # 효과: 야간 화재(어두움) ~
    #       주간 화재(밝음) 대응
    hsv_v=0.5,

    # ── 학습 안정화 설정 ──────────────────────────

    # [lr0] 초기 학습률
    # 기본값: 0.01 유지
    # 이유: 5000장 규모에 적합
    lr0=0.01,

    # [lrf] 최종 학습률 비율
    # 기본값: 0.01 유지 (lr0의 1%까지 감소)
    # 이유: 학습 후반 미세 조정
    lrf=0.01,

    # [warmup_epochs] 웜업 에폭 수
    # 기본값: 3.0 유지
    # 효과: 처음 3에폭 동안 학습률 천천히 증가
    #       급격한 학습 방지
    warmup_epochs=3.0,

    # [weight_decay] 가중치 감쇠 (과적합 방지)
    # 기본값: 0.0005 유지
    # 효과: 학습 데이터에만 과도하게 맞춰지는 것 방지
    weight_decay=0.0005,

    # ── 모자이크 OFF 시점 ─────────────────────────

    # [close_mosaic] 마지막 N 에폭에서 모자이크 끄기
    # 기본값: 10 유지
    # 이유: 학습 초반은 모자이크로 다양성
    #       마지막 10 에폭은 실제 이미지로 미세 조정
    #       이렇게 해야 실제 추론 시 성능 향상
    close_mosaic=10,
)

# 종료 시간 기록
end_time = time.time()

# 소요 시간 계산 (초 단위)
elapsed = end_time - start_time

# 완료 메시지 출력
print("=" * 60)
print(f"✅ 학습 완료!")
# 종료 시각 출력
print(f"종료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# 소요 시간 출력 (분, 시간 단위)
print(f"소요 시간: {elapsed/60:.1f}분 ({elapsed/3600:.2f}시간)")
print("=" * 60)


[하이퍼파라미터 튜닝 학습 시작]
시작 시각: 2026-03-03 21:48:08

【튜닝 내용】
  1. Augmentation 강화 (scale, mosaic, copy_paste, mixup)
  2. 색상 증강 강화 (hsv_h, hsv_s, hsv_v)
  3. 기하학적 증강 추가 (degrees, translate, flipud)
  4. Patience 조정 (50 → 15)
Ultralytics 8.4.19  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\data_70_15.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, li

셀 5: 최종 결과 확인

In [5]:
# ============================================
# 최종 결과 파일 확인
# ============================================

print("\n[최종 결과]")
print("=" * 60)

# Best 모델 파일 경로 (가장 성능 좋은 모델)
best_model = RESULTS_DIR / 'weights' / 'best.pt'

# Best 모델 파일 존재 확인
if best_model.exists():
    # 파일이 있으면 경로 출력
    print(f"✅ Best 모델: {best_model}")
    
    # 모델 파일 크기 확인 (MB 단위)
    size_mb = best_model.stat().st_size / 1024**2
    print(f"   크기: {size_mb:.2f} MB")
else:
    # 파일이 없으면 경고
    print(f"⚠️ Best 모델 없음")

# 학습 결과 CSV 파일 확인 (에폭별 성능 기록)
results_csv = RESULTS_DIR / 'results.csv'
if results_csv.exists():
    print(f"✅ 결과 CSV: {results_csv}")

# Confusion Matrix 이미지 확인
confusion_matrix = RESULTS_DIR / 'confusion_matrix.png'
if confusion_matrix.exists():
    print(f"✅ Confusion Matrix: {confusion_matrix}")

# 학습 그래프 이미지 확인
results_png = RESULTS_DIR / 'results.png'
if results_png.exists():
    print(f"✅ 학습 그래프: {results_png}")

# 전체 결과 폴더 위치 출력
print(f"\n📂 전체 결과 위치:")
print(f"   {RESULTS_DIR}")

print("=" * 60)


[최종 결과]
✅ Best 모델: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\weights\best.pt
   크기: 5.97 MB
✅ 결과 CSV: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\results.csv
✅ Confusion Matrix: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\confusion_matrix.png
✅ 학습 그래프: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\results.png

📂 전체 결과 위치:
   N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned


셀 6: Validation 성능 확인

In [ ]:
# ============================================
# Validation 성능 확인
# ============================================

print("\n[Validation 성능]")
print("=" * 60)

# Best 모델 파일 로드
best_model_path = RESULTS_DIR / 'weights' / 'best.pt'
best_model = YOLO(str(best_model_path))

# Validation 데이터로 성능 평가 실행
val_results = best_model.val(
    data=str(DATA_YAML),   # 데이터 설정 파일 (val 경로 포함)
    imgsz=640,              # 이미지 크기
    batch=16,               # 배치 크기
    device=0                # GPU 0번 사용
)

# 주요 성능 지표 출력
print(f"\n📊 주요 지표:")
# mAP50: IoU 50%에서의 평균 정밀도
print(f"  mAP50:     {val_results.box.map50:.3f}")
# mAP50-95: IoU 50-95%에서의 평균 정밀도
print(f"  mAP50-95:  {val_results.box.map:.3f}")
# Precision: 정밀도 (예측한 것 중 맞은 비율)
print(f"  Precision: {val_results.box.mp:.3f}")
# Recall: 재현율 (실제 중 찾은 비율)
print(f"  Recall:    {val_results.box.mr:.3f}")

print("\n【기존 v8n과 비교】")
print(f"                기존(0.20)  튜닝 후")
print(f"  mAP50:        0.755       {val_results.box.map50:.3f}")
print(f"  Recall:       0.696       {val_results.box.mr:.3f}")
print(f"  Precision:    0.787       {val_results.box.mp:.3f}")

print("=" * 60)
print("🎉 하이퍼파라미터 튜닝 완료!")
print("\n다음 단계: Test Set으로 최종 평가")
print("=" * 60)




# ## 📁 저장되는 파일

# N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\
# ├── weights/
# │   ├── best.pt          ← 최고 성능 모델
# │   └── last.pt          ← 마지막 에폭 모델
# ├── results.csv          ← 에폭별 성능
# ├── results.png          ← 학습 그래프
# ├── confusion_matrix.png ← 혼동 행렬
# └── ...


[Validation 성능]
Ultralytics 8.4.19  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 48.212.6 MB/s, size: 209.4 KB)
val: Scanning N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15\val\labels.cache... 893 images, 430 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 894/894  0.0s
val: N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\val\images\fire_0230.jpg: ignoring corrupt image/label: cannot identify image file 'N:\\\\\\3.13_Mini_Project\\DATASET\\5000_split_70_15\\val\\images\\fire_0230.jpg'
val: N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\val\images\fire_0509.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.08692]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 15.2it/s 3.7s0.0s
                   all        892       1154      0.7